In [0]:
from pyspark.sql import functions as F
SILVER = "olist_project.silver"
GOLD = "olist_project.gold"
results = []

In [0]:
null_checks = {
   f"{SILVER}.orders":        ["order_id", "customer_id", "order_status"],
   f"{SILVER}.order_items":   ["order_id", "product_id", "price"],
   f"{SILVER}.order_payments":["order_id", "payment_value"],
   f"{SILVER}.customers":     ["customer_id", "customer_state"],
   f"{SILVER}.products":      ["product_id"],
   f"{GOLD}.revenue_by_month":["year", "month", "total_revenue"],
   f"{GOLD}.customer_segments":["customer_unique_id", "customer_type"]
}
print(f"{'Table':<40} {'Column':<30} {'Null Count':>10} {'Status':>10}")
print("-" * 95)
for table, columns in null_checks.items():
   df = spark.table(table)
   for col in columns:
       null_count = df.filter(F.col(col).isNull()).count()
       status = "PASS" if null_count == 0 else "FAIL"
       results.append({"check": "null_check", "table": table, "column": col,
                       "value": null_count, "status": status})
       print(f"{table:<40} {col:<30} {null_count:>10} {status:>10}")

Table                                    Column                         Null Count     Status
-----------------------------------------------------------------------------------------------
olist_project.silver.orders              order_id                                0       PASS
olist_project.silver.orders              customer_id                             0       PASS
olist_project.silver.orders              order_status                            0       PASS
olist_project.silver.order_items         order_id                                0       PASS
olist_project.silver.order_items         product_id                              0       PASS
olist_project.silver.order_items         price                                   0       PASS
olist_project.silver.order_payments      order_id                                0       PASS
olist_project.silver.order_payments      payment_value                           0       PASS
olist_project.silver.customers           customer_id      

In [0]:

row_count_checks = {
   f"{SILVER}.orders":          90000,
   f"{SILVER}.customers":       90000,
   f"{SILVER}.order_items":    100000,
   f"{SILVER}.order_payments": 100000,
   f"{GOLD}.revenue_by_month":     20,
   f"{GOLD}.revenue_by_state":     25,
   f"{GOLD}.product_performance": 30000,
   f"{GOLD}.customer_segments":   90000
}
print(f"{'Table':<45} {'Expected':>10} {'Actual':>10} {'Status':>10}")
print("-" * 80)
for table, min_expected in row_count_checks.items():
   actual = spark.table(table).count()
   status = "PASS" if actual >= min_expected else "FAIL"
   results.append({"check": "row_count_check", "table": table,
                   "value": actual, "status": status})
   print(f"{table:<45} {min_expected:>10} {actual:>10} {status:>10}")

Table                                           Expected     Actual     Status
--------------------------------------------------------------------------------
olist_project.silver.orders                        90000      99441       PASS
olist_project.silver.customers                     90000      99441       PASS
olist_project.silver.order_items                  100000     112650       PASS
olist_project.silver.order_payments               100000     103886       PASS
olist_project.gold.revenue_by_month                   20         22       PASS
olist_project.gold.revenue_by_state                   25         27       PASS
olist_project.gold.product_performance             30000      32216       PASS
olist_project.gold.customer_segments               90000      93395       PASS


In [0]:
duplicate_checks = {
   f"{SILVER}.orders":    "order_id",
   f"{SILVER}.customers": "customer_id",
   f"{SILVER}.products":  "product_id",
   f"{SILVER}.sellers":   "seller_id"
}
print(f"{'Table':<40} {'Key Column':<25} {'Duplicates':>10} {'Status':>10}")
print("-" * 90)
for table, key_col in duplicate_checks.items():
   df        = spark.table(table)
   total     = df.count()
   distinct  = df.select(key_col).distinct().count()
   dupe_count = total - distinct
   status    = "PASS" if dupe_count == 0 else "FAIL"
   results.append({"check": "duplicate_check", "table": table,
                   "column": key_col, "value": dupe_count, "status": status})
   print(f"{table:<40} {key_col:<25} {dupe_count:>10} {status:>10}")

Table                                    Key Column                Duplicates     Status
------------------------------------------------------------------------------------------
olist_project.silver.orders              order_id                           0       PASS
olist_project.silver.customers           customer_id                        0       PASS
olist_project.silver.products            product_id                         0       PASS
olist_project.silver.sellers             seller_id                          0       PASS


In [0]:
print("=== Business Logic Checks ===\n")
# 1. No negative revenue
neg_revenue = spark.table(f"{SILVER}.order_payments") \
   .filter(F.col("payment_value") < 0).count()
status = "PASS" if neg_revenue == 0 else "FAIL"
print(f"No negative payment values:        {status} ({neg_revenue} found)")
results.append({"check": "business_logic", "table": f"{SILVER}.order_payments",
               "column": "payment_value", "value": neg_revenue, "status": status})
# 2. Review scores only between 1-5
bad_scores = spark.table(f"{SILVER}.order_reviews") \
   .filter(~F.col("review_score").between(1, 5)).count()
status = "PASS" if bad_scores == 0 else "FAIL"
print(f"Review scores between 1-5:         {status} ({bad_scores} found)")
results.append({"check": "business_logic", "table": f"{SILVER}.order_reviews",
               "column": "review_score", "value": bad_scores, "status": status})
# 3. No future order dates
future_orders = spark.table(f"{SILVER}.orders") \
   .filter(F.col("order_purchase_timestamp") > F.current_timestamp()).count()
status = "PASS" if future_orders == 0 else "FAIL"
print(f"No future order timestamps:        {status} ({future_orders} found)")
results.append({"check": "business_logic", "table": f"{SILVER}.orders",
               "column": "order_purchase_timestamp", "value": future_orders, "status": status})
# 4. Customer segments are valid values only
invalid_segments = spark.table(f"{GOLD}.customer_segments") \
   .filter(~F.col("customer_type").isin("High Value", "Returning", "One Time")).count()
status = "PASS" if invalid_segments == 0 else "FAIL"
print(f"Valid customer segment values:     {status} ({invalid_segments} found)")
results.append({"check": "business_logic", "table": f"{GOLD}.customer_segments",
               "column": "customer_type", "value": invalid_segments, "status": status})

=== Business Logic Checks ===

No negative payment values:        PASS (0 found)
Review scores between 1-5:         PASS (0 found)
No future order timestamps:        PASS (0 found)
Valid customer segment values:     PASS (0 found)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
   StructField("check",  StringType(),  True),
   StructField("table",  StringType(),  True),
   StructField("column", StringType(),  True),
   StructField("value",  StringType(),  True),
   StructField("status", StringType(),  True)
])
results_clean = [{**r, "value": str(r.get("value", "")),
                     "column": r.get("column", "")} for r in results]
results_df = spark.createDataFrame(results_clean, schema=schema)
total  = results_df.count()
passed = results_df.filter(F.col("status") == "PASS").count()
failed = results_df.filter(F.col("status") == "FAIL").count()
print("=" * 50)
print(f"  DATA QUALITY REPORT")
print("=" * 50)
print(f"  Total Checks : {total}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print("=" * 50)
if failed > 0:
   print("\nFailed Checks:")
   results_df.filter(F.col("status") == "FAIL").show(truncate=False)
else:
   print("\nAll checks passed — pipeline is healthy!")

  DATA QUALITY REPORT
  Total Checks : 43
  Passed       : 43
  Failed       : 0

All checks passed — pipeline is healthy!
